# Credenciales de kaggle y descarga del dataset

In [2]:
!kaggle datasets download -d shanegerami/ai-vs-human-text -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text
License(s): other
100% 350M/350M [00:02<00:00, 148MB/s]



# Exploracion del dataset


In [15]:
import pandas as pd

df = pd.read_csv('/content/data/AI_Human.csv') #1 - Generated by IA . 0 Generated by Human
print(df.shape)
df.head()
print(df['generated'].value_counts())


(487235, 2)
generated
0.0    305797
1.0    181438
Name: count, dtype: int64


# Limpieza basica del dataset

In [10]:
df = df.dropna(subset=['text', 'generated'])

# Asegurar tipo correcto en label
df['generated'] = df['generated'].astype(int)

# Revisar longitud de textos
df['text_len'] = df['text'].str.len()
print(df['text_len'].describe())

# Filtrar textos con menos de 100 caracteres
df = df[df['text_len'] >= 100]

print(f"Filas después de filtrar: {df.shape[0]}")
print(df['generated'].value_counts())


count    487203.000000
mean       2269.732502
std         988.682544
min         102.000000
25%        1583.000000
50%        2102.000000
75%        2724.000000
max       18322.000000
Name: text_len, dtype: float64
Filas después de filtrar: 487203
generated
0    305796
1    181407
Name: count, dtype: int64


#  Features con TF-IDF + Split

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Entrenamiento: 80% - Test: 20%
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['generated'],
    test_size=0.2,
    random_state=42,
    stratify=df['generated']
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# Vectorizer
vectorizer = TfidfVectorizer(max_features=50000, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Shape del train vectorizado: {X_train_tfidf.shape}")

Train: 389762 | Test: 97441
Shape del train vectorizado: (389762, 50000)


# Entrenamiento del modelo


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import time

# Entrenar
print("Entrenando...")
start = time.time()

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='saga',
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

print(f"Entrenado en {time.time() - start:.1f} segundos")

Entrenando...
Entrenado en 43.2 segundos


# Evaluación del modelo


In [13]:
# Predicciones
y_pred = model.predict(X_test_tfidf)

# Reporte completo
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))

# Matriz de confusión
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       Human       1.00      1.00      1.00     61159
          AI       1.00      1.00      1.00     36282

    accuracy                           1.00     97441
   macro avg       1.00      1.00      1.00     97441
weighted avg       1.00      1.00      1.00     97441

Matriz de confusión:
[[61036   123]
 [  171 36111]]


# Guardar modelo y vectorizer


In [19]:
# Ver cómo son los textos humanos del dataset
sample = df[df['generated'] == 1]['text'].iloc[5]
print(sample[:500])

  A Sustainable Urban Future  Car-free cities are emerging as a powerful response to the pressing challenges of urbanization. These cities aspire to create environments where private automobiles are either severely restricted or completely banned, emphasizing sustainable transportation alternatives, cleaner air, and vibrant urban living. This essay delves into the concept of car-free cities, exploring their potential benefits, challenges, and solutions.  Car-free cities are gaining momentum as a


In [14]:
import joblib

joblib.dump(model, '/content/model.pkl')
joblib.dump(vectorizer, '/content/vectorizer.pkl')

print("Archivos guardados:")
!ls -lh /content/*.pkl

Archivos guardados:
-rw-r--r-- 1 root root 392K Mar 31 17:28 /content/model.pkl
-rw-r--r-- 1 root root 1.8M Mar 31 17:28 /content/vectorizer.pkl
